# GitNexus Agent - Ask Questions About Your Codebase

This notebook demonstrates an agentic structure that uses OpenAI as the LLM and GitNexus tools to answer questions about the microservices-demo codebase.

**Architecture:**
- LLM: OpenAI (gpt-4o or gpt-4)
- Tools: GitNexus (query, context, impact, cypher)
- Agent: Simple tool-calling loop with reasoning

## Setup & Configuration

In [108]:
import subprocess
import json
import os
from pathlib import Path
from typing import Any, Optional
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env file
load_dotenv()

# Configure OpenAI
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found. Please set it in .env file or as an environment variable.")

client = OpenAI(api_key=api_key)

print(f"✓ OpenAI API key loaded")
print(f"✓ Using model: gpt-4o-mini")

# Repository working directory
REPO_DIR = Path.cwd()
print(f"\n✓ Working directory: {REPO_DIR}")

# Verify GitNexus connection
result = subprocess.run(
    "gitnexus status",
    shell=True,
    capture_output=True,
    text=True,
    cwd=REPO_DIR
)
print(f"\nGitNexus Status:\n{result.stdout}")

✓ OpenAI API key loaded
✓ Using model: gpt-4o-mini

✓ Working directory: c:\Users\EwegConsulting\Documents\Projects\gitnexus docker\gitnexus\repos\microservices-demo

GitNexus Status:
Repository: c:\Users\EwegConsulting\Documents\Projects\gitnexus docker\gitnexus\repos\microservices-demo
Indexed: 14-4-2026, 11:28:52
Indexed commit: c9857ee
Current commit: c9857ee
Status: âœ… up-to-date



## GitNexus Tool Wrappers

In [109]:
# Tool definitions for OpenAI function calling
TOOL_DEFINITIONS = [
    {
        "type": "function",
        "function": {
            "name": "gitnexus_context",
            "description": "Get detailed context about a specific code symbol (function, class, method). Shows callers, callees, and process participation. Best for: understanding specific functions like 'PlaceOrder', 'chargeCard', etc.",
            "parameters": {
                "type": "object",
                "properties": {
                    "symbol": {
                        "type": "string",
                        "description": "Symbol name or full UID (e.g., 'PlaceOrder', 'chargeCard', 'Method:src/checkoutservice/main.go:PlaceOrder')"
                    }
                },
                "required": ["symbol"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "gitnexus_impact",
            "description": "Analyze the blast radius of changes to a code symbol. Shows what breaks or is affected if you modify this symbol.",
            "parameters": {
                "type": "object",
                "properties": {
                    "target": {
                        "type": "string",
                        "description": "Symbol to analyze"
                    },
                    "direction": {
                        "type": "string",
                        "enum": ["upstream", "downstream"],
                        "description": "upstream: who calls this; downstream: what this calls"
                    }
                },
                "required": ["target"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "gitnexus_cypher",
            "description": "Execute a Neo4j Cypher query directly against the code graph. Use for complex graph analysis and dependency searches.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Cypher query string for graph analysis"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

print("Tool definitions created (3 tools)")
print(f"Available tools: {[t['function']['name'] for t in TOOL_DEFINITIONS]}")
print("Tools: gitnexus_context (symbols), gitnexus_impact (blast radius), gitnexus_cypher (graph queries)")

Tool definitions created (3 tools)
Available tools: ['gitnexus_context', 'gitnexus_impact', 'gitnexus_cypher']
Tools: gitnexus_context (symbols), gitnexus_impact (blast radius), gitnexus_cypher (graph queries)


## Agent Implementation

In [110]:
# System prompt to guide the agent's querying strategy
SYSTEM_PROMPT = """You are an expert code analysis agent with access to GitNexus tools to analyze a microservices codebase.

GITNEXUS GRAPH SCHEMA:
Node types: File, Method, Function, Class, Constructor, Interface, Struct, Property, Process, Section, Route, Folder, Namespace, Community, CodeEmbedding
Key attributes:
  - File: filePath, name, content (full file text)
  - Method/Function/Class: name, filePath, startLine, endLine
  - Process: id, summary (text description of process flow)
  - Relationships: CONTAINS, CALLS, REFERENCES, IMPORTS, EXTENDS, IMPLEMENTS

TOOL SELECTION STRATEGY:

**Use gitnexus_cypher for:**
1. Specific framework/dependency searches - find files containing a string
   Example: MATCH (f:File) WHERE f.content CONTAINS 'log4j' RETURN f.filePath, f.name
2. Finding all usages of a symbol across the codebase
   Example: MATCH (m:Method {name: 'PlaceOrder'})-[:CALLS]->(other) RETURN other.name
3. Searching by file patterns
   Example: MATCH (f:File) WHERE f.filePath CONTAINS 'build.gradle' RETURN f.filePath
4. Complex graph queries with multiple hops
   Example: MATCH (p1:Process)-[:CONTAINS]->(f1:Function)-[:CALLS]->(f2:Function) RETURN p1.id, f2.name

**Use gitnexus_context for:**
- Understanding a specific symbol once you know its name
- Getting callers, callees, and process participation
- Example: context("PlaceOrder")

**Use gitnexus_impact for:**
- Analyzing what breaks if you change a symbol
- Finding all dependents upstream or downstream
- Example: impact("ChargeCard", direction="upstream")

IMPORTANT CYPHER TIPS:
- Use f.content CONTAINS 'string' to search file contents (case-sensitive)
- Use f.filePath CONTAINS 'pattern' to search file paths
- Node types must exist: File, Function, Method, Class, Process, Interface, Struct, etc. (NOT Literal, Import, or invented types)
- Return specific attributes: RETURN f.filePath, f.name (NOT just RETURN f)
- Limit results to avoid huge result sets: LIMIT 10"""

In [111]:
from dataclasses import dataclass
import json

@dataclass
class Message:
    """Represents a message in the conversation"""
    role: str  # 'user', 'assistant', 'tool'
    content: str | list = ""


class CodebaseAgent:
    """Agent that uses OpenAI to answer questions about the codebase using GitNexus tools"""
    
    def __init__(self, model: str = "gpt-4o-mini", max_iterations: int = 25, max_result_chars: int = 5000):
        """
        Initialize the agent.
        
        Args:
            model: OpenAI model to use
            max_iterations: Maximum number of tool calls before stopping
            max_result_chars: Maximum characters per tool result
        """
        self.model = model
        self.max_iterations = max_iterations
        self.max_result_chars = max_result_chars
        self.conversation_history = []
        
    def run(self, question: str) -> str:
        """
        Run the agent with a user question.
        
        Args:
            question: The user's question about the codebase
            
        Returns:
            The agent's final response
        """
        # Initialize conversation with system prompt first
        self.conversation_history = [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": question
            }
        ]
        
        separator = "=" * 70
        print()
        print(separator)
        print(f"Question: {question}")
        print(f"Model: {self.model}")
        print(separator)
        print()
        
        for iteration in range(self.max_iterations):
            print(f"[Iteration {iteration + 1}] Calling OpenAI...")
            
            # Get response from OpenAI (system prompt is in messages)
            try:
                response = client.chat.completions.create(
                    model=self.model,
                    messages=self.conversation_history,
                    tools=TOOL_DEFINITIONS,
                    tool_choice="auto",
                    temperature=0.5
                )
            except Exception as e:
                print(f"ERROR calling OpenAI: {e}")
                return f"Error: {e}"
            
            assistant_message = response.choices[0].message
            print(f"  Response received")
            
            # Debug: Show if there are tool calls
            if assistant_message.tool_calls:
                print(f"  Has {len(assistant_message.tool_calls)} tool call(s)")
            else:
                print(f"  No tool calls - finishing")
            
            # Check if we're done (no tool calls)
            if not assistant_message.tool_calls:
                # Add final response to history
                self.conversation_history.append({
                    "role": "assistant",
                    "content": assistant_message.content or ""
                })
                print()
                print(separator)
                print("FINAL ANSWER:")
                print(separator)
                print(assistant_message.content)
                return assistant_message.content
            
            # Add assistant message with tool calls to history
            self.conversation_history.append({
                "role": "assistant",
                "content": assistant_message.content or "",
                "tool_calls": [
                    {
                        "id": tc.id,
                        "type": "function",
                        "function": {
                            "name": tc.function.name,
                            "arguments": tc.function.arguments
                        }
                    }
                    for tc in assistant_message.tool_calls
                ]
            })
            
            # Show thinking
            if assistant_message.content:
                print(f"  Thinking: {assistant_message.content[:80]}...")
            
            # Execute tool calls
            for tool_call in assistant_message.tool_calls:
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments)
                
                print(f"  Executing: {tool_name}")
                
                try:
                    # Execute the appropriate tool
                    if tool_name == "gitnexus_query":
                        result = gitnexus_query(tool_args["query"])
                    elif tool_name == "gitnexus_context":
                        result = gitnexus_context(tool_args["symbol"])
                    elif tool_name == "gitnexus_impact":
                        result = gitnexus_impact(
                            tool_args["target"],
                            tool_args.get("direction", "upstream")
                        )
                    elif tool_name == "gitnexus_cypher":
                        result = gitnexus_cypher(tool_args["query"])
                    else:
                        result = {"error": f"Unknown tool: {tool_name}"}
                    
                    # Format result
                    result_str = json.dumps(result, indent=2)[:self.max_result_chars]
                    print(f"    Got result ({len(result_str)} chars)")
                    
                except Exception as e:
                    result_str = f"Error: {str(e)}"
                    print(f"    Error: {e}")
                
                # Add tool result message
                self.conversation_history.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result_str
                })
        
        # Max iterations reached - return best answer so far
        print()
        print(separator)
        print("REACHED MAX ITERATIONS - PRESENTING PARTIAL RESULTS")
        print(separator)
        print()
        
        # Extract the last assistant response with content
        last_response = None
        for msg in reversed(self.conversation_history):
            if msg.get("role") == "assistant" and msg.get("content"):
                last_response = msg.get("content")
                break
        
        if last_response:
            print(last_response)
            return last_response
        
        # Fallback: Show last few tool results
        tool_results = [msg for msg in self.conversation_history if msg.get("role") == "tool"]
        if tool_results:
            num_results = len(tool_results)
            print(f"Agent conducted {num_results} tool searches.")
            print("Last results collected:")
            for result in tool_results[-3:]:
                content = result.get("content", "")
                if len(content) > 500:
                    print(content[:500])
                    print("...")
                else:
                    print(content)
                print()
            return "See results above"
        
        return "Agent reached maximum iterations"
print("CodebaseAgent defined")

CodebaseAgent defined


## Usage - Ask Your Question Here

In [112]:
# Initialize the agent
agent = CodebaseAgent(model="gpt-4.1-mini", max_iterations=10, max_result_chars=50000)

# ============================================================================
# SPECIFY YOUR QUESTION HERE
# ============================================================================
# question = "Which services depend on PaymentService and what would break if we changed its API? Try to investigate how the API is actually used in the dependant entities"
question = "Explain the architecture of the microservices-demo repo. What are the main services and how do they interact?"
# question = "What are the most common patterns used in error handling across the codebase?"
# question = "Is it safe to refactor CartService? What services depend on it? Check the usage in those dependencies"
# question = "Which applications in this repo use Log4j and what version?"
# ============================================================================
# Run the agent
answer = agent.run(question)


Question: Explain the architecture of the microservices-demo repo. What are the main services and how do they interact?
Model: gpt-4.1-mini

[Iteration 1] Calling OpenAI...
  Response received
  Has 1 tool call(s)
  Executing: gitnexus_cypher
    Got result (24275 chars)
[Iteration 2] Calling OpenAI...
  Response received
  No tool calls - finishing

FINAL ANSWER:
The microservices-demo repo, also known as Online Boutique, is a cloud-first microservices demo application that simulates a web-based e-commerce app. Users can browse items, add them to the cart, and purchase them. The application is composed of 11 microservices written in different languages that communicate over gRPC.

Main Services:
1. frontend (Go): Exposes an HTTP server to serve the website. It generates session IDs for users automatically.
2. cartservice (C#): Stores and retrieves items in the user's shopping cart using Redis.
3. productcatalogservice (Go): Provides the list of products from a JSON file and supports 

## Conversation History

In [113]:
# Display the conversation history for debugging/understanding
import json

print("\nConversation History:")
print("="*70)
for i, msg in enumerate(agent.conversation_history, 1):
    role = msg.get("role", "unknown").upper()
    content = msg.get("content", "")
    
    if isinstance(content, str):
        preview = content[:200] + "..." if len(content) > 200 else content
        print(f"{i}. [{role}] {preview}")
    elif isinstance(content, list):
        print(f"{i}. [{role}] Tool results ({len(content)} items)")
    else:
        print(f"{i}. [{role}] {str(content)[:100]}")
    print()


Conversation History:
1. [SYSTEM] You are an expert code analysis agent with access to GitNexus tools to analyze a microservices codebase.

GITNEXUS GRAPH SCHEMA:
Node types: File, Method, Function, Class, Constructor, Interface, Stru...

2. [USER] Explain the architecture of the microservices-demo repo. What are the main services and how do they interact?

3. [ASSISTANT] 

4. [TOOL] {
  "markdown": "| f.filePath | f.content |\n| --- | --- |\n| README.md | <!-- <p align=\"center\">\n<img src=\"/src/frontend/static/icons/Hipster_HeroLogoMaroon.svg\" width=\"300\" alt=\"Online Bouti...

5. [ASSISTANT] The microservices-demo repo, also known as Online Boutique, is a cloud-first microservices demo application that simulates a web-based e-commerce app. Users can browse items, add them to the cart, and...



## Agent Architecture Overview

```
┌─────────────────────────────────────────────────────────────────┐
│                     User Question                                 │
└────────────────────────────┬────────────────────────────────────┘
                             │
                             ▼
┌─────────────────────────────────────────────────────────────────┐
│                  OpenAI Chat Completion                          │
│              (gpt-4o with tool calling)                         │
└────────────────────────────┬────────────────────────────────────┘
                             │
                    ┌────────┴────────┐
                    ▼                 ▼
         ┌──────────────────┐  ┌──────────────────┐
         │  Reasoning Text  │  │   Tool Calls     │
         └──────────────────┘  └────────┬─────────┘
                                        │
                    ┌───────────────────┼───────────────────┐
                    ▼                   ▼                   ▼
          ┌──────────────────┐  ┌──────────────────┐  ┌──────────────────┐
          │ gitnexus_query   │  │gitnexus_context  │  │ gitnexus_impact  │
          │    (Search)      │  │  (Symbol Info)   │  │  (Blast Radius)  │
          └────────┬─────────┘  └────────┬─────────┘  └────────┬─────────┘
                   │                     │                     │
                   └─────────────────────┼─────────────────────┘
                                         │
                                         ▼
                          ┌──────────────────────────┐
                          │   GitNexus Code Graph    │
                          │   (261 files, 2128      │
                          │    symbols, 5089 edges) │
                          └──────────────────────────┘
                                         │
                    ┌────────────────────┼────────────────────┐
                    ▼                    ▼                    ▼
             ┌──────────────┐    ┌──────────────┐    ┌──────────────┐
             │ Search Results│   │ Call Graph   │    │Process Flows │
             │ & Processes   │   │& Dependencies│    │& Execution  │
             └──────────────┘    └──────────────┘    └──────────────┘
                    │                    │                    │
                    └────────────────────┼────────────────────┘
                                         │
                                         ▼
                          ┌──────────────────────────┐
                          │   Agent Reasoning Loop   │
                          │  (refine → tool calls →  │
                          │   analyze → repeat)      │
                          └──────────────────────────┘
                                         │
                                         ▼
                          ┌──────────────────────────┐
                          │  Final Answer to User    │
                          │ (with code locations &   │
                          │  explanations)           │
                          └──────────────────────────┘
```